In [ ]:
%%capture
# see comments in README on changes to the conda venv
import os
from pathlib import Path

# import modin.pandas as pd
import pandas as pd
from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

## Appendix II. Outcome indicators according to single and multiple health conditions

In [ ]:
import numpy as np
from intecomm_analytics.dataframes import get_df_main_1858
import statsmodels.api as sm
import statsmodels.formula.api as smf
from math import exp

In [ ]:
df_main_orig = get_df_main_1858(None, fasting_hours=8.0)


In [ ]:
df_main = df_main_orig[df_main_orig.retained_12m==1].copy()
df_main = df_main.reset_index(drop=True)
assert len(df_main.query("primary_cohort.isin([1,2,3,4])")) == 1600

df_main["assignment"] = df_main.assignment.apply(lambda s: "intervention" if s =="a" else "control")
df_main['assignment'] = pd.Categorical(df_main['assignment'])


In [ ]:
df1 = (
    df_main
    .groupby(["assignment", "primary_cohort_str"], observed=True)
    .size()
    .reset_index()
    .pivot_table(columns="assignment", index="primary_cohort_str", values=0, observed=True)
)
df1.columns.name = "condition"
df1.index.name=""
df1["total"] = df1.intervention + df1.control
df1


In [ ]:
df_main["primary_cohort_str"] = df_main.apply(lambda x: "HTN_DM_HIV" if x.primary_cohort_str=='UNDEFINED' and x.hiv==1 and x.htn==1 and x.dm==1 else x.primary_cohort_str, axis=1)
df_main["primary_cohort_str"] = df_main.apply(lambda x: "HTN_HIV" if x.primary_cohort_str=='UNDEFINED' and x.hiv==1 and x.htn==1 and x.dm==0 else x.primary_cohort_str, axis=1)
df_main["primary_cohort_str"] = df_main.apply(lambda x: "DM_HIV" if x.primary_cohort_str=='UNDEFINED' and x.hiv==1 and x.htn==0 and x.dm==1 else x.primary_cohort_str, axis=1)
df_tot = df_main.groupby(["assignment", "primary_cohort_str"], observed=True).size().reset_index().pivot_table(columns="assignment", index="primary_cohort_str", values=0, observed=True)
df_tot["total"] = df_tot.control + df_tot.intervention

df_tot.columns.name = "condition"
df_tot.index.name=""
df_tot

In [ ]:
def get_rows_for_outcome(binary_col:str, cond_str:str):
    df1 = df_main.query(cond_str).groupby(["primary_cohort_str", "assignment"], observed=True).agg(outcome=(binary_col, "sum"), total=(binary_col, "size")).reset_index()
    df1["outcome%"] = np.round((df1["outcome"] / df1["total"]) * 100, 1)
    # df1["assignment"] = df1.assignment.apply(lambda s: "c" if s =="a" else "f")
    df_pivot = df1.pivot(index="primary_cohort_str", columns="assignment", values=["total", "outcome", "outcome%"])
    df_pivot.columns = [f'{col[0]}_{col[1]}' for col in df_pivot.columns if col != "primary_cohort_str"]
    for col in df_pivot.columns :
        df_pivot[col] = df_pivot[col].astype("Float64").round(1)
    return df_pivot

def get_rows_for_value(value_col:str, cond_str:str):
    df2 = df_main.query(cond_str).groupby(["primary_cohort_str", "assignment"], observed=True).agg(mean=(value_col, "mean"), std=(value_col, "std")).reset_index()
    # df2["assignment"] = df2.assignment.apply(lambda s: "c" if s =="a" else "f")
    df_pivot2 = df2.pivot(index="primary_cohort_str", columns="assignment", values=["mean", "std"])
    df_pivot2.columns = [f'{col[0]}_{col[1]}' for col in df_pivot2.columns if col[0] != "primary_cohort_str"]
    for col in df_pivot2.columns :
        df_pivot2[col] = df_pivot2[col].astype("Float64").round(1)
    return df_pivot2

def get_rr(col:str, primary_cohort_str:str, adjust:bool, family:str|None=None):
    family = "poisson" if family is None else family
    if adjust:
        formula = f"{col} ~ assignment + age_in_years + C(gender)"
    else:
        formula = f"{col} ~ assignment"

    if family=="poisson":
        model = smf.glm(
            formula=formula,
            data=df_main.query(f"primary_cohort_str==@primary_cohort_str and ~{col}.isna()")[["assignment", "gender", "age_in_years", col]],
            family=sm.families.Poisson()
        )
        results = model.fit(cov_type='HC0')
    elif family=="log-binomial":
        model = smf.glm(
            formula=formula,
            data=df_main.query(f"primary_cohort_str==@primary_cohort_str and ~{col}.isna()")[["assignment", "gender", "age_in_years", col]],
            family=sm.families.Binomial(link=sm.families.links.Log())
        )
        results = model.fit(cov_type='HC0', maxiter=50)
    else:
        raise ValueError(f"family {family} not supported")


    rr_term = [term for term in results.params.index if 'assignment' in term and 'intervention' in term][0]

    p_value = results.pvalues[rr_term]

    # Coefficient (Log(RR))
    log_rr = results.params[rr_term]

    # Confidence Interval (Log scale)
    ci_log = results.conf_int().loc[rr_term].values

    rr = exp(log_rr)
    rr_lower = exp(ci_log[0])
    rr_upper = exp(ci_log[1])

    return rr, rr_lower, rr_upper, p_value, results, rr_term


def get_rr_crude_and_adjusted(col:str, primary_cohort_str:str, family:str|None=None):
    family = "poisson" if family is None else family
    unadjusted_rr, unadjusted_rr_lower, unadjusted_rr_upper, unadjusted_p_value, unadjusted_results, unadjusted_rr_term = get_rr(col, primary_cohort_str, adjust=False, family=family)
    adjusted_rr, adjusted_rr_lower, adjusted_rr_upper, adjusted_p_value, adjusted_results, adjusted_rr_term = get_rr(col, primary_cohort_str, adjust=True, family=family)

    df = pd.DataFrame({
    "primary_cohort_str": primary_cohort_str,
    "crude": [unadjusted_rr],
    "crude_ci_lower": [unadjusted_rr_lower],
    "crude_ci_upper": [unadjusted_rr_upper],
    "crude_p_value": [unadjusted_p_value],
    "adjusted": [adjusted_rr],
    "adjusted_ci_lower": [adjusted_rr_lower],
    "adjusted_ci_upper": [adjusted_rr_upper],
    "adjusted_p_value": [adjusted_p_value],
    },
    index=[primary_cohort_str]
    )
    df = df[[col for col in df.columns if col != "primary_cohort_str"]].round(3)
    confounding_detected = abs(unadjusted_rr - adjusted_rr) > 0.1 * unadjusted_rr
    if confounding_detected:
        print(f"\n{col} for {primary_cohort_str}: 'age' and 'gender' are possible confounders.")
    else:
        pass
        # print(f"\n{col}: 'age' and 'gender' not strong confounders.")
    return df


### bp_controlled_endline

In [ ]:
col =  "bp_controlled_endline"
cond = "htn==1"
df = get_rows_for_outcome(col, cond)
df_t = df.T.reset_index().rename(columns={"index":"variables"}).copy()
df_t.index.name = ""
df_t.columns.name = ""
df_t.query("variables.str.endswith('intervention')")

In [ ]:
df_t.query("variables.str.endswith('control')")

In [ ]:
dfs = []
for primary_cohort_str in df.index.values:
    dfs.append(get_rr_crude_and_adjusted(col, primary_cohort_str))
df_rr = pd.concat(dfs, ignore_index=False)
df_rr_t = df_rr.T.reset_index().rename(columns={"index":"variables"}).copy()
df_rr_t.index.name = ""
df_rr_t.columns.name = ""
for col in df_rr_t.columns:
    if col != "variables":
        df_rr_t[col] = df_rr_t[col].round(2)
df_rr_t.query("variables.str.startswith('crude')")

In [ ]:
df_rr_t.query("variables.str.startswith('adjusted')")

#### bp_sys_endline mean (std)

In [ ]:
col =  "bp_sys_endline"
cond = "htn==1"
df1 = get_rows_for_value(col, f"{cond} and assignment=='intervention'")
df2 = get_rows_for_value(col, f"{cond} and assignment=='control'")
df = df1.merge(df2, on="primary_cohort_str")
df_t = df.T
df_t = df_t.reset_index().rename(columns={"index":"variables"}).copy()
df_t.columns.name = ""
df_t.query("variables.str.endswith('intervention')")

In [ ]:
df_t.query("variables.str.endswith('control')")


In [ ]:
dfs = []
for primary_cohort_str in df.index.values:
    dfs.append(get_rr_crude_and_adjusted(col, primary_cohort_str))
df_rr = pd.concat(dfs, ignore_index=False)
df_rr_t = df_rr.T.reset_index().rename(columns={"index":"variables"}).copy()
df_rr_t.index.name = ""
df_rr_t.columns.name = ""
for col in df_rr_t.columns:
    if col != "variables":
        df_rr_t[col] = df_rr_t[col].round(2)
df_rr_t.query("variables.str.startswith('crude')")

In [ ]:
df_rr_t.query("variables.str.startswith('adjusted')")

#### bp_dia_endline mean (std)


In [ ]:
col =  "bp_dia_endline"
cond = "htn==1"
df1 = get_rows_for_value(col, f"{cond} and assignment=='intervention'")
df2 = get_rows_for_value(col, f"{cond} and assignment=='control'")
df = df1.merge(df2, on="primary_cohort_str")
df_t = df.T
df_t = df_t.reset_index().rename(columns={"index":"variables"}).copy()
df_t.columns.name = ""
df_t.query("variables.str.endswith('intervention')")

In [ ]:
df_t.query("variables.str.endswith('control')")

In [ ]:
dfs = []
for primary_cohort_str in df.index.values:
    dfs.append(get_rr_crude_and_adjusted(col, primary_cohort_str))
df_rr = pd.concat(dfs, ignore_index=False)
df_rr_t = df_rr.T.reset_index().rename(columns={"index":"variables"}).copy()
df_rr_t.index.name = ""
df_rr_t.columns.name = ""
for col in df_rr_t.columns:
    if col != "variables":
        df_rr_t[col] = df_rr_t[col].round(2)
df_rr_t.query("variables.str.startswith('crude')")

In [ ]:
df_rr_t.query("variables.str.startswith('adjusted')")

### glucose_controlled_endline

In [ ]:
col =  "glucose_controlled_endline"
cond = "dm==1"
df = get_rows_for_outcome(col, cond)
df_t = df.T.reset_index().rename(columns={"index":"variables"}).copy()
df_t.index.name = ""
df_t.columns.name = ""
df_t.query("variables.str.endswith('intervention')")

In [ ]:
df_t.query("variables.str.endswith('control')")


In [ ]:
dfs = []
for primary_cohort_str in df.index.values:
    dfs.append(get_rr_crude_and_adjusted(col, primary_cohort_str))
df_rr = pd.concat(dfs, ignore_index=False)
df_rr_t = df_rr.T.reset_index().rename(columns={"index":"variables"}).copy()
df_rr_t.index.name = ""
df_rr_t.columns.name = ""
for col in df_rr_t.columns:
    if col != "variables":
        df_rr_t[col] = df_rr_t[col].round(2)
df_rr_t.query("variables.str.startswith('crude')")

In [ ]:
df_rr_t.query("variables.str.startswith('adjusted')")


#### glucose_value_endline mean (std)

In [ ]:
col =  "glucose_value_endline"
cond = "dm==1"
df1 = get_rows_for_value(col, f"{cond} and assignment=='intervention'")
df2 = get_rows_for_value(col, f"{cond} and assignment=='control'")
df = df1.merge(df2, on="primary_cohort_str")
df_t = df.T
df_t = df_t.reset_index().rename(columns={"index":"variables"}).copy()
df_t.columns.name = ""
df_t.query("variables.str.endswith('intervention')")

In [ ]:
df_t.query("variables.str.endswith('control')")

In [ ]:
dfs = []
for primary_cohort_str in df.index.values:
    dfs.append(get_rr_crude_and_adjusted(col, primary_cohort_str))
df_rr = pd.concat(dfs, ignore_index=False)
df_rr_t = df_rr.T.reset_index().rename(columns={"index":"variables"}).copy()
df_rr_t.index.name = ""
df_rr_t.columns.name = ""
for col in df_rr_t.columns:
    if col != "variables":
        df_rr_t[col] = df_rr_t[col].round(2)
df_rr_t.query("variables.str.startswith('crude')")

In [ ]:
df_rr_t.query("variables.str.startswith('adjusted')")


### HIV

In [ ]:
col =  "vl_controlled_endline"
cond = "hiv==1"
df = get_rows_for_outcome(col, cond)
df_t = df.T.reset_index().rename(columns={"index":"variables"}).copy()
df_t.index.name = ""
df_t.columns.name = ""
df_t.query("variables.str.endswith('intervention')")

In [ ]:
df_t.query("variables.str.endswith('control')")


In [ ]:
dfs = []
for primary_cohort_str in df.index.values:
    dfs.append(get_rr_crude_and_adjusted(col, primary_cohort_str))
df_rr = pd.concat(dfs, ignore_index=False)
df_rr_t = df_rr.T.reset_index().rename(columns={"index":"variables"}).copy()
df_rr_t.index.name = ""
df_rr_t.columns.name = ""
for col in df_rr_t.columns:
    if col != "variables":
        df_rr_t[col] = df_rr_t[col].round(2)
df_rr_t.query("variables.str.startswith('crude')")

In [ ]:
# import pandas as pd
# import numpy as np
# from scipy.stats import fisher_exact
# from math import log, exp
#
#
# # Define the groups for the comparison
# group_A = 'intervention'
# group_B = 'control'
# col = "bp_controlled_endline"
# EVENT_NAME = 'BP Controlled' # The positive outcome (1)
#
# # --- 2. Calculate Counts for the Risk Ratio (Contingency Table) ---
#
# # Aggregate the counts
# contingency_table = df_main.query("primary_cohort_str.isin(['HTN_DM_HIV'])").groupby('assignment', observed=True)[col].agg(
#     events_count='sum',
#     total_count='count'
# ).reset_index()
#
# # Calculate the number of people who did NOT have the event (0)
# contingency_table['no_event_count'] = contingency_table['total_count'] - contingency_table['events_count']
#
# # Extract the values for the Risk Ratio formula
# events_A = contingency_table.loc[contingency_table['assignment'] == group_A, 'events_count'].iloc[0]
# total_A = contingency_table.loc[contingency_table['assignment'] == group_A, 'total_count'].iloc[0]
# no_events_A = total_A - events_A
#
# events_B = contingency_table.loc[contingency_table['assignment'] == group_B, 'events_count'].iloc[0]
# total_B = contingency_table.loc[contingency_table['assignment'] == group_B, 'total_count'].iloc[0]
# no_events_B = total_B - events_B
#
# # Check for division by zero risk (if either total_A or total_B is 0, which is unlikely)
# if total_A == 0 or total_B == 0:
#     print("Error: One or both groups have a total count of zero.")
#
# # --- 3. Calculate Risk Ratio (RR) ---
#
# # Risk (Proportion) in Group A
# risk_A = events_A / total_A
# # Risk (Proportion) in Group B
# risk_B = events_B / total_B
#
# # Handle division by zero if risk_B is zero (very rare, but possible if no events in control)
# if risk_B == 0:
#     risk_ratio = float('inf')
# else:
#     risk_ratio = risk_A / risk_B
#
# # --- 4. Print Results and Interpretation ---
#
# print(f"--- Risk Ratio Analysis: '{EVENT_NAME}' Outcome ---")
# print(f"Group: {group_A} (A) | Controlled: {events_A} | Total: {total_A} | Rate: {risk_A:.3f}")
# print(f"Group: {group_B} (B) | Controlled: {events_B} | Total: {total_B} | Rate: {risk_B:.3f}")
# print("-" * 50)
# print(f"Calculated Risk Ratio (RR): {risk_ratio:.2f}")
#
# # --- Interpretation ---
# if risk_ratio > 1:
#     print(f"\nInterpretation: The rate of achieving '{EVENT_NAME}' is {risk_ratio:.2f} times higher in the {group_A} group compared to the {group_B} group.")
#     print(f"This means Group A is {((risk_ratio - 1) * 100):.1f}% more likely to be controlled.")
# elif risk_ratio < 1:
#     print(f"\nInterpretation: The rate of achieving '{EVENT_NAME}' is lower in the {group_A} group.")
#     print(f"Group A is only {risk_ratio:.2f} times as likely as Group B to be controlled.")
# else:
#     print("\nInterpretation: The rate of control is approximately the same between the two groups (RR ~ 1).")
#
# # Optional: Calculating the Confidence Interval (Requires statistical library for robust calculation)
# # For simplicity, we use the log-based approximation method for CI:
# def calculate_rr_ci(e1, n1, e2, n2):
#     p1 = e1 / n1
#     p2 = e2 / n2
#
#     # Standard Error of the log of the Risk Ratio
#     se_log_rr = ((1 / e1) - (1 / n1) + (1 / e2) - (1 / n2)) ** 0.5
#
#     # 95% Confidence Interval for the log RR
#     log_rr = log(p1 / p2)
#     lower_log_rr = log_rr - 1.96 * se_log_rr
#     upper_log_rr = log_rr + 1.96 * se_log_rr
#
#     # Convert back to standard scale
#     rr_lower = exp(lower_log_rr)
#     rr_upper = exp(upper_log_rr)
#     return rr_lower, rr_upper
#
# if risk_A > 0 and risk_B > 0 and events_A > 0 and events_B > 0:
#     rr_lower, rr_upper = calculate_rr_ci(events_A, total_A, events_B, total_B)
#     print(f"\n95% Confidence Interval for RR: ({rr_lower:.3f}, {rr_upper:.3f})")
#
# # Fisher's exact test (for hypothesis testing on the difference)
# table_data = [[events_A, no_events_A], [events_B, no_events_B]]
# odds_ratio, p_value = fisher_exact(table_data)
#
# print("-" * 50)
# print(f"Statistical Note (Fisher's Exact Test P-value): {p_value:.4f}")